# LATAM Economic Dashboard — Exploratory Data Analysis

This notebook performs EDA on economic indicators for six Latin American countries
(Chile, Argentina, Brazil, Mexico, Colombia, Peru) sourced from the World Bank API.

**Author:** Eduardo Moraga  
**Indicators:** GDP per Capita, GDP Growth, Inflation, Trade (% GDP), Merchandise Exports, Unemployment, Internet Users  
**Period:** 2000–2024

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.load import load_table
from src.utils import COUNTRIES, INDICATORS, INDICATOR_SHORT, COUNTRY_COLORS

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Load Data

We load both the raw long-format data and the transformed wide-format data from SQLite.
Make sure you have run the ETL pipeline first (via the Streamlit app or manually).

In [ ]:
raw = load_table("raw_indicators")
transformed = load_table("transformed")

print(f"Raw data: {raw.shape[0]:,} records")
print(f"Transformed data: {transformed.shape[0]:,} rows x {transformed.shape[1]} columns")
raw.head()

## 2. Data Quality Analysis

In [ ]:
# Missing values by country and indicator
raw["value"] = pd.to_numeric(raw["value"], errors="coerce")

coverage = raw.pivot_table(
    index="country_name",
    columns="indicator_name",
    values="value",
    aggfunc="count",
)

fig = px.imshow(
    coverage,
    text_auto=True,
    color_continuous_scale="Greens",
    title="Data Coverage: Number of Non-null Years per Country/Indicator",
    labels={"color": "Years with data"},
)
fig.update_layout(height=400, width=900)
fig.show()

In [ ]:
# Overall missing rate
total_possible = len(COUNTRIES) * len(INDICATORS) * 25  # 25 years
total_present = raw["value"].notna().sum()
print(f"Overall coverage: {total_present}/{total_possible} ({100*total_present/total_possible:.1f}%)")
print()

# Missing by indicator
missing_by_ind = raw.groupby("indicator_name")["value"].apply(lambda s: s.isna().sum())
print("Missing values by indicator:")
print(missing_by_ind.sort_values(ascending=False))

## 3. Distribution Analysis

Box plots showing the distribution of each indicator across all countries and years.

In [ ]:
fig = px.box(
    raw.dropna(subset=["value"]),
    x="indicator_name",
    y="value",
    color="country_name",
    color_discrete_map=COUNTRY_COLORS,
    title="Distribution of Indicators by Country",
)
fig.update_layout(height=500, xaxis_tickangle=-30)
fig.show()

In [ ]:
# Descriptive statistics for the transformed wide-format data
indicator_cols = [c for c in INDICATOR_SHORT.values() if c in transformed.columns]
transformed[indicator_cols].describe().round(2)

## 4. Trend Visualizations

Time series for each indicator with all six countries overlaid.

In [ ]:
for ind_code, ind_name in INDICATORS.items():
    subset = raw[raw["indicator"] == ind_code].dropna(subset=["value"])
    if subset.empty:
        continue
    fig = px.line(
        subset,
        x="year",
        y="value",
        color="country_name",
        markers=True,
        title=ind_name,
        color_discrete_map=COUNTRY_COLORS,
        labels={"value": ind_name, "year": "Year", "country_name": "Country"},
    )
    fig.update_layout(height=400, width=900)
    fig.show()

## 5. Correlation Heatmap

How do the different economic indicators correlate with each other across all countries?

In [ ]:
indicator_cols = [c for c in INDICATOR_SHORT.values() if c in transformed.columns]
corr = transformed[indicator_cols].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlation Matrix of Economic Indicators",
)
fig.update_layout(height=500, width=600)
fig.show()

## 6. GDP Growth vs Other Indicators (Scatter Matrix)

In [ ]:
scatter_cols = [c for c in ["GDP Growth", "Inflation", "Unemployment", "Trade", "Internet"] if c in transformed.columns]

if scatter_cols:
    fig = px.scatter_matrix(
        transformed.dropna(subset=scatter_cols),
        dimensions=scatter_cols,
        color="country_name",
        color_discrete_map=COUNTRY_COLORS,
        title="Scatter Matrix: Key Economic Indicators",
        opacity=0.6,
    )
    fig.update_layout(height=800, width=900)
    fig.update_traces(diagonal_visible=False)
    fig.show()

## 7. Year-over-Year Volatility

Which countries show the most economic volatility?

In [ ]:
yoy_cols = [c for c in transformed.columns if c.endswith("_yoy")]

if yoy_cols:
    volatility = transformed.groupby("country_name")[yoy_cols].std()
    volatility.columns = [c.replace("_yoy", "") for c in volatility.columns]
    
    fig = px.bar(
        volatility.reset_index().melt(id_vars="country_name"),
        x="variable",
        y="value",
        color="country_name",
        barmode="group",
        color_discrete_map=COUNTRY_COLORS,
        title="YoY Volatility (Std Dev of % Change) by Country",
        labels={"value": "Std Dev (%)", "variable": "Indicator"},
    )
    fig.update_layout(height=450, width=900)
    fig.show()

## 8. Key Insights

### GDP per Capita
- Chile has consistently the highest GDP per capita in the group, typically 2-3x that of Peru and Colombia.
- All countries show a positive long-term trend with notable dips during the 2008 financial crisis and the 2020 COVID-19 pandemic.

### Inflation
- Argentina is a clear outlier with inflation rates frequently exceeding 30-50%, while the other five countries have largely maintained single-digit or low-double-digit inflation.
- Chile, Mexico, and Peru demonstrate successful inflation targeting regimes.

### Trade & Integration
- Chile and Mexico lead in trade openness (trade as % of GDP), reflecting their extensive free-trade agreement networks.
- Brazil, despite being the largest economy, has relatively low trade openness due to its large domestic market.

### Digital Adoption
- Internet penetration shows a convergence trend — all countries have moved from 5-15% to 70-85% over the 2000-2024 period.
- Chile leads, while Peru has closed the gap significantly in recent years.

### Correlations
- GDP per capita correlates strongly with internet penetration (digital infrastructure tends to accompany development).
- High inflation negatively correlates with GDP growth, particularly for Argentina.
- Trade openness shows a moderate positive correlation with GDP per capita.

In [ ]:
print("EDA complete. See the Streamlit app for interactive exploration.")